### Estimating Dipole Moment of CP Radical

Using the pyscf and pyberny packages. Code generated by Gemini Pro 3, edited and commented by BAM.

In [1]:
import pyscf
from pyscf import dft
from pyscf.geomopt.berny_solver import optimize 
from pyscf.hessian import thermo

In [2]:
#Define the CP radical molecule
mol = pyscf.M(
    atom='C 0.0 0.0 0.0; P 0.0 0.0 1.562',
    #basis='6-311++g(d,p)',
    basis='aug-cc-pVTZ',
    spin=1,
    charge=0,
    unit='Angstrom'
)

In [3]:
#Build the Unrestricted DFT object (UKS) and set the functional
mf = dft.UKS(mol)
mf.xc = 'm06-2x'

In [4]:
#Perform the Geometry Optimization
print("--- Starting Geometry Optimization (M06-2X / aug-cc-pVTZ) ---")
mol_eq = optimize(mf)

--- Starting Geometry Optimization (M06-2X / aug-cc-pVTZ) ---

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   P   0.000000   0.000000   1.562000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -379.287108208538  <S^2> = 0.80439056  2S+1 = 2.0536704
--------------- UKS_Scanner gradients ---------------
         x                y                z
0 C     0.0000000000     0.0000000000    -0.0165515557
1 P    -0.0000000000    -0.0000000000     0.0166341583
----------------------------------------------
cycle 1: E = -379.287108209  dE = -379.287  norm(grad) = 0.0234659

Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   0.000000   0.000000   0.008419    0.000000  0.000000  0.008419
   P   0.000000   0.00

In [5]:
print("\nOptimized Geometry Structure:")
print(mol_eq.tostring())


Optimized Geometry Structure:
C           0.00000000        0.00000000        0.00797136
P           0.00000000        0.00000000        1.55402864


In [6]:
#Run a final single-point calculation on the optimized structure.
# 'optimize()' returns a new Mole object (mol_eq) with updated coordinates.
print("\n--- Running Final Calculation on Optimized Geometry ---")
mf_eq = dft.UKS(mol_eq)
mf_eq.xc = 'm06-2x'
mf_eq.kernel()


--- Running Final Calculation on Optimized Geometry ---
converged SCF energy = -379.287360317807  <S^2> = 0.79946774  2S+1 = 2.0488707


-379.2873603178066

In [7]:
#Extract and print the final dipole moment
print("\n--- Final Dipole Moment Results ---")
dipole_vector = mf_eq.dip_moment()


--- Final Dipole Moment Results ---
Dipole moment(X, Y, Z, Debye):  0.00000,  0.00000,  0.90487


In [8]:
#Compute Rotational Constants for comparison to lab work (B = 23859.914 from Saito:1989:1114)
print("\n=== ROTATIONAL CONSTANT ===")
# PySCF stores coordinates in Bohr internally; thermo.rotation_const expects Bohr and AMU
masses = mol_eq.atom_mass_list()
coords = mol_eq.atom_coords()

# Calculate the constants in MHz
rot_constants = thermo.rotation_const(masses, coords, unit='GHz')*1000.
print(f'A: {rot_constants[0]:.2f} MHz\nB: {rot_constants[1]:.2f} MHz\nC: {rot_constants[2]:.2f} MHz')


=== ROTATIONAL CONSTANT ===
A: inf MHz
B: 24439.46 MHz
C: 24439.46 MHz


Given that these are B_e values, this is close enough to the Saito work for the dipole to be a reasonable estimate.